# CodeSentinel real training (Colab)\nUse a GPU runtime. This notebook produces serving artifacts; do not treat the demo bundle as a real detector.

In [ ]:
# T4 16 GB assumption: batch_size=8, grad accumulation=2, epochs=3.\n# Reduce these if the available GPU/session is smaller.\n!git clone <YOUR_REPOSITORY_URL> codesentinel-model\n%cd codesentinel-model\n!pip install -r requirements.txt

In [ ]:
!python data/download.py --dataset all --out data/raw\n!python data/preprocess.py --raw-dir data/raw --out data/processed/all_samples.jsonl\n!python data/dedup_split.py --in data/processed/all_samples.jsonl --out-dir data/processed/

In [ ]:
# Edit training/config.yaml for microsoft/graphcodebert-base, epochs: 3, batch_size: 8.\n!python training/train.py --config training/config.yaml\n!python evaluation/generate_report.py --config training/config.yaml

In [ ]:
!python export/export_onnx.py --checkpoint training/final_model/model.pt --config training/config.yaml --out codesentinel_model/artifacts/model.onnx\n!python export/export_torchscript.py --checkpoint training/final_model/model.pt --config training/config.yaml --out codesentinel_model/artifacts/model_ts.pt\n!python export/parity_check.py --pytorch-model training/final_model/model.pt --onnx-model codesentinel_model/artifacts/model.onnx --torchscript-model codesentinel_model/artifacts/model_ts.pt --config training/config.yaml --include-padding\n!python export/benchmark.py --pytorch-model training/final_model/model.pt --onnx-model codesentinel_model/artifacts/model.onnx --torchscript-model codesentinel_model/artifacts/model_ts.pt --config training/config.yaml

In [ ]:
# Save the real tokenizer into the artifact bundle, then zip it for download.\nfrom transformers import AutoTokenizer\nAutoTokenizer.from_pretrained('microsoft/graphcodebert-base').save_pretrained('codesentinel_model/artifacts/tokenizer')\n!zip -r codesentinel-serving-artifacts.zip codesentinel_model/artifacts